# Preprocessing Technique: Handling Missing Data
**Member:** [Fernando GVN/ IT25101718]
**Technique:** Handling Missing Data
**Dataset:** Tourism Recommendation Dataset (100,000 rows)


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("tourism_recommendation_dataset_en.csv")
print("Shape:", df.shape)
df.head()

Shape: (100000, 25)


,tourist_id,gender,age,age_group,source_province,attraction_name,attraction_category,attraction_level,city,province,...,rating,is_group_tour,group_fee,trip_days,main_spots,transport_mode,season,is_holiday,recommendation_level,satisfaction_level
0,1,Female,67,56+,Beijing,Xuan Kong Si,Religious Culture,4A,Da Tong Shi,Shanxi,...,4.3,No,NaN,NaN,NaN,NaN,Winter,No,Recommend,Satisfied
1,2,Male,22,18-25,Heilongjiang,Kai Feng Fu,Historical Culture,4A,Kai Feng Shi,Henan,...,4.8,No,NaN,NaN,NaN,NaN,Autumn,No,Highly Recommend,Very Satisfied
2,3,Male,60,56+,Fujian,Zhou Zhuang Gu Zhen,Ancient Town,5A,Su Zhou Shi,Jiangsu,...,4.4,No,NaN,NaN,NaN,NaN,Summer,No,Recommend,Satisfied
3,4,Female,33,26-35,Ningxia,Zhu Jia Jiao Gu Zhen,Ancient Town,4A,Shanghai,Shanghai,...,3.7,No,NaN,NaN,NaN,NaN,Autumn,No,Recommend,Satisfied
4,5,Female,59,56+,Jiangxi,Lu Shan,Natural Culture,5A,Jiu Jiang Shi,Jiangxi,...,4.9,No,NaN,NaN,NaN,NaN,Winter,No,Highly Recommend,Very Satisfied


### Prerequisite check: duplicate rows


In [15]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Duplicate rows found: 0
Shape after removing duplicates: (100000, 25)


In [16]:
# Step 1: Quantify missing values
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_summary

,missing_count,missing_pct
group_fee,70242,70.24
trip_days,70242,70.24
main_spots,70242,70.24
transport_mode,70242,70.24


In [17]:
# Step 2: Confirm the missingness is tied to is_group_tour (structural, not random)
for col in ["group_fee", "trip_days", "transport_mode", "main_spots"]:
    print(col, "-> missing rate broken down by is_group_tour:")
    print(df.groupby("is_group_tour")[col].apply(lambda s: s.isnull().mean().round(3)))
    print()

group_fee -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: group_fee, dtype: float64

trip_days -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: trip_days, dtype: float64

transport_mode -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: transport_mode, dtype: float64

main_spots -> missing rate broken down by is_group_tour:
is_group_tour
No     1.0
Yes    0.0
Name: main_spots, dtype: float64



In [18]:
# Step 3: Handle the missingness

df_clean = df.copy()
df_clean["is_group_tour_flag"] = (df_clean["is_group_tour"] == "Yes").astype(int)

for col in ["group_fee", "trip_days"]:
    group_median = df_clean.loc[df_clean["is_group_tour"] == "Yes", col].median()
    df_clean[col] = df_clean[col].fillna(group_median)

for col in ["transport_mode", "main_spots"]:
    df_clean[col] = df_clean[col].fillna("Not Applicable")

print("Remaining missing values:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Remaining missing values:
 Series([], dtype: int64)


In [19]:
# Step 4: Verify no unintended data loss
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
assert df_clean.shape[0] == df.shape[0], "Row count should not change — we imputed, not dropped" 

Original shape: (100000, 25)
Cleaned shape: (100000, 26)


## Generating output

In [20]:
df_clean.to_csv("output_after_missing_data.csv", index=False)
print("Saved output_after_missing_data.csv - shape:", df_clean.shape)


Saved output_after_missing_data.csv - shape: (100000, 26)
